In [30]:
import qnexus as qnx
#qnx.logout()
qnx.login()


🌐 Browser log in initiated.


╭────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                        │
│         Confirm that the browser shows the following code and click 'allow device':    │
│                                                                                        │
│                                      63WEAW                                            │
│                                                                                        │
╰────────────────────────────────────────────────────────────────────────────────────────╯

Browser didn't open automatically? Use this link: https://nexus.quantinuum.com/auth/device/browser?otp=63WEAWvWM2TCBiZyMlHbIabA176wjVDLVcv0iWU_0lIvW_pQAOn3vdAkkWKRxyZKKGIeIvTAKACM6RyUw5r59A
✅ Successfully logged in as tranminh1831996@gmail.com using the browser.


In [ ]:
import qnexus as qnx
from pytket.circuit import Circuit
from pytket.circuit.display import render_circuit_jupyter

circuit = Circuit(10)
circuit.H(0)
for i, j in zip(circuit.qubits[:-1], circuit.qubits[1:]):
    circuit.CX(i, j)
circuit.measure_all();
from pytket.circuit.display import render_circuit_jupyter

render_circuit_jupyter(circuit)
ref = qnx.circuits.upload(circuit=circuit, name=f"GHZ-Circuit")
render_circuit_jupyter(ref.download_circuit())

config = qnx.QuantinuumConfig(device_name="H1-Emulator")
ref_compile_job = qnx.start_compile_job(
    programs=[ref],
    backend_config=config,
    optimisation_level=2,
    name=f"compilation-job",
)


In [ ]:
import qnexus as qnx
from pytket.circuit import Circuit
from pytket.circuit.display import render_circuit_jupyter

import datetime

config = qnx.QuantinuumConfig(device_name="H1-Emulator")

project = qnx.projects.get_or_create("MCMR")
qnx.context.set_active_project(project)
name_suffix = datetime.datetime.now().strftime("%Y_%m_%d")
config = qnx.QuantinuumConfig(device_name="H1-Emulator")

# circuit
circuit_test = Circuit(2,2)
circuit_test.H(0)
circuit_test.CX(0,1)
circuit_test.measure_all()


# circuit must be updload into nexus database before compilation

ref_circuit = qnx.circuits.upload(
    circuit=circuit_test,
    name=f"repetition-code-circuit-{name_suffix}",
    description="TKET circuit  code"
)


# complie jobs
ref_compile_job = qnx.start_compile_job(
    circuits=[ref_circuit], # use the one uploaded
    optimisation_level=2,
    backend_config=config,
    name=f"job2"
)



qnx.jobs.wait_for(ref_compile_job)
ref_compiled_circuit = qnx.jobs.results(ref_compile_job)[0].get_output()
compiled_circuit = ref_compiled_circuit.download_circuit()
render_circuit_jupyter(compiled_circuit)

# excute jobs

ref_execute_job = qnx.start_execute_job(
    programs=[ref_compiled_circuit],
    n_shots=[100],
    backend_config=config,
    name=f"execution-job-{name_suffix}",
)

qnx.jobs.status(ref_execute_job)
qnx.jobs.wait_for(ref_execute_job)
ref_result = qnx.jobs.results(ref_execute_job)[0]
backend_result = ref_result.download_result()
backend_result.get_distribution()



c:\Users\tranm\anaconda3\Lib\site-packages\qnexus\client\utils.py:170: DeprecationWarning: The `circuits` argument is deprecated and will be removed in a future version. Please use `programs`.
  warnings.warn(


C:\Users\tranm\AppData\Local\Temp\ipykernel_39280\565791729.py:57: DeprecationWarning: The `BackendResult.get_distribution()` method is deprecated: please use `get_empirical_distribution()` or `get_probability_distribution()` instead.
  backend_result.get_distribution()


{(0, 0): 0.6, (1, 1): 0.4}

In [ ]:
def apply_unitary(wires):
    # Define Hermitian matrix A
    A = np.array([
        [1, 0, 0, 1],
        [0, 2, 1, 0],
        [0, 1, 3, 0],
        [1, 0, 0, 4]
    ], dtype=np.complex128)
    # Compute the unitary U = exp(iA)
    U = expm(1j * A)
    # Confirm unitarity
    assert np.allclose(U.conj().T @ U, np.eye(4)), "U is not unitary"
    # Apply the unitary operation to the specified wires
    qml.QubitUnitary(U, wires=wires)

# Define the quantum phase estimation circuit
def qpe_circuit(phase_wires, target_wire): 
    # Initialize target in eigenstate of U
    # qml.Hadamard(wires=target_wire)

    # Apply Hadamards to phase register
    for w in phase_wires:
        qml.Hadamard(wires=w)

    # Apply controlled-U^{2^k}
    apply_unitary_powers = [2 ** i for i in range(len(phase_wires))]

    for i, power in enumerate(apply_unitary_powers):
        for _ in range(power):
            qml.ctrl(apply_unitary, control=phase_wires[::-1][i])(wires=target_wire)

    # Apply inverse QFT
    qml.adjoint(qml.QFT)(wires=phase_wires)

def AQE_fractional_rotation(control_wires, target_wire):
    """Apply controlled RY rotation based on fractional binary interpretation of control qubits."""
    n = len(control_wires)

    for d in range(1, 2**n):  # skip d=0 to avoid divide-by-zero
        # Convert decimal to binary string to run all possible state
        bin_str = f"{d:0{n}b}"
        # Convert binary string to list of bits for controlled rotation
        bit_strings = [int(bit) for bit in bin_str]
        # Compute fractional binary value: d = sum x_j * 2^-j
        fractional_d = sum(int(bit) * 2**-(j+1) for j, bit in enumerate(bin_str))
        # Compute the inverse of the fractional value
        inv = 1 / fractional_d
        # Normalize to [0, 1] range
        inv = inv / (2 ** n)
        # Convert to angle for RY gate, the state |0> amplitude corresponds to inv value
        theta = np.arccos(inv) * 2  # 2*theta for full RY
        # Apply controlled RY rotation on target wire controling on control wires
        qml.ctrl(qml.RY, control=control_wires, control_values=bit_strings)(theta,wires=target_wire)

dev = qml.device("default.qubit", wires=tot_qubits)
@qml.qnode(dev)
def hhl_circuit():
    # 1. Initialize the target qubit
    for target_wire in target_wires:
        qml.Hadamard(wires=target_wire) # in this example, |b> = U_b |0> = H_0 H_1 ... |0>

    # 2. Apply QPE (Quantum Phase Estimation)
    qpe_circuit(phase_wires, target_wires)

    # 3. Apply AQE fractional rotation (controlled RY based on phase_wires)
    AQE_fractional_rotation(phase_wires, ancilla_wires)

    # 4. Apply inverse QPE (adjoint of QPE)
    qml.adjoint(qpe_circuit)(phase_wires, target_wires)

    # 5. Measure the target qubit
    probs = qml.probs(wires= [*target_wires,*ancilla_wires])
    
    return probs

In [ ]:
pip install Aer

In [10]:
pip install qiskit-aer


Note: you may need to restart the kernel to use updated packages.


In [8]:

import numpy as np
from scipy.linalg import expm
from qiskit_aer import AerSimulator
from qiskit import QuantumCircuit
from qiskit.circuit.library import QFT, RYGate, UnitaryGate
from qiskit.quantum_info import Statevector
import json

# Define Hermitian matrix A
A = np.array([
    [1, 0, 0, 1],
    [0, 2, 1, 0],
    [0, 1, 3, 0],
    [1, 0, 0, 4]
], dtype=np.complex128)

# Compute the unitary U = exp(iA)
U = expm(1j * A)
assert np.allclose(U.conj().T @ U, np.eye(4)), "U is not unitary"

# Define main quantum circuit
n_phase = 2           # Number of phase estimation qubits
n_target = 2          # Dimension of U = exp(iA) => 2 qubits
n_ancilla = 1
n_total = n_phase + n_target + n_ancilla

qc = QuantumCircuit(n_total)

phase_wires = list(range(n_phase))
target_wires = list(range(n_phase, n_phase + n_target))
ancilla_wire = n_total - 1

# Step 1: Initialize target qubits in |b> (Hadamard on each)
for qubit in target_wires:
    qc.h(qubit)

# Step 2: Apply Hadamard to phase register
qc.h(phase_wires)

# Step 3: Apply controlled-U^{2^k}
unitary_gate = UnitaryGate(U)
for i, k in enumerate(reversed(phase_wires)):
    for _ in range(2 ** i):
        controlled_gate = unitary_gate.control(num_ctrl_qubits=1)
        qc.append(controlled_gate, [k] + target_wires)

# Step 4: Inverse QFT
qc.append(QFT(len(phase_wires), inverse=True).decompose(), phase_wires)

# Step 5: Apply AQE fractional RY rotation on ancilla
for d in range(1, 2 ** n_phase):
    bin_str = f"{d:0{n_phase}b}"
    fractional_d = sum(int(bit) * 2 ** -(j + 1) for j, bit in enumerate(bin_str))
    inv = 1 / fractional_d
    inv /= (2 ** n_phase)
    theta = 2 * np.arccos(inv)

    # Build control logic
    controls = []
    ctrl_bits = []
    for idx, bit in enumerate(bin_str):
        controls.append(phase_wires[idx])
        ctrl_bits.append(int(bit))

    # Apply controlled RY to ancilla based on control bits
    qc.mcry(theta, controls, ancilla_wire, mode='noancilla')  # Assumes no ancilla is used

# Step 6: Apply adjoint of QPE (roughly reverse QPE steps)
qc.append(QFT(len(phase_wires)).decompose(), phase_wires)  # QFT
for i, k in enumerate(reversed(phase_wires)):
    for _ in range(2 ** i):
        controlled_gate = unitary_gate.control(num_ctrl_qubits=1)
        qc.append(controlled_gate.inverse(), [k] + target_wires)

qc.h(phase_wires)

# Step 7: Measurement
qc.measure_all()

# Simulate

job = AerSimulator().run(qc,shots=1024)
result = job.result().get_counts(qc)
result_json = json.dumps(result)
print("Measurement outcomes:", result_json)


C:\Users\tranm\AppData\Local\Temp\ipykernel_32112\725550510.py:48: DeprecationWarning: The class ``qiskit.circuit.library.basis_change.qft.QFT`` is deprecated as of Qiskit 2.1. It will be removed in Qiskit 3.0. ('Use qiskit.circuit.library.QFTGate or qiskit.synthesis.qft.synth_qft_full instead, for access to all previous arguments.',)
  qc.append(QFT(len(phase_wires), inverse=True).decompose(), phase_wires)
C:\Users\tranm\AppData\Local\Temp\ipykernel_32112\725550510.py:69: DeprecationWarning: The class ``qiskit.circuit.library.basis_change.qft.QFT`` is deprecated as of Qiskit 2.1. It will be removed in Qiskit 3.0. ('Use qiskit.circuit.library.QFTGate or qiskit.synthesis.qft.synth_qft_full instead, for access to all previous arguments.',)
  qc.append(QFT(len(phase_wires)).decompose(), phase_wires)  # QFT


AerError: 'unknown instruction: c-unitary'